# 01 — Prepare the shared dataset

Builds, once, everything the model notebooks need, then saves it as one Kaggle Dataset
shared with the team (see `docs/dataprep.md`).

| Step | What | Output in `asr-data/` |
|---|---|---|
| 1 | FLEURS `vi_vn`, VIVOS, VietMed → 16 kHz, filtered, normalized, split | `clean/`, `manifests/clean.csv` |
| 2 | Traffic (FSD50K) and voices (MUSAN), split train/test by file | `noise/`, `manifests/noise_*.csv` |
| 3 | Noisy test (9 conditions) and dev sets, with SNR QC; training schedule | `noisy/`, `manifests/*_plan.csv`, `manifests/train_schedule.csv` |
| 4 | Final checks, `DATASET.json`, `LICENSES.md` | |

**Kaggle settings**: CPU session, *Internet on*, input dataset `asr-noisy-vi-code`
attached. Expected run time 1–3 h (MUSAN is read from an 11 GB archive). Code and
downloads go to `/tmp`, outside the saved output.

**Resume**: every step skips files that already exist; if the session stops, run all
cells again.

**Dry run first**: set `LIMIT = 20` to process a few files per split in minutes, check the
output, then set `LIMIT = None`, delete `/kaggle/working/asr-data` and run for real.

**Share**: when the run ends, open the notebook version → *Output* → *New Dataset*, name it
`asr-noisy-vi`, keep it **private** and share it with the team (*Settings → Sharing*).

## 0. Setup

The code comes from the Kaggle Dataset **`asr-noisy-vi-code`**, made on a local machine
with `python scripts/package_code.py` and uploaded as a zip. Attach it with *Add Input*.
Its `CODE_VERSION.json` (commit, branch) is copied into `DATASET.json`.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
CODE_DATASET = None  # folder of the attached code; None = search /kaggle/input


def find_code(root=Path("/kaggle/input")):
    """The attached folder holding the packages (the zip may be extracted one level deeper)."""
    for depth in range(1, 6):
        pattern = "/".join(["*"] * depth) + "/dataprep/__init__.py"
        for hit in sorted(root.glob(pattern)):
            if (hit.parents[1] / "pyproject.toml").is_file():
                return hit.parents[1]
    raise FileNotFoundError("Code not found: attach the Kaggle Dataset 'asr-noisy-vi-code' "
                            "(Add Input → Datasets → Your Datasets), or set CODE_DATASET.")


if Path("dataprep").exists():  # running from inside the repo (local machine)
    code_dir = Path.cwd()
else:
    source = Path(CODE_DATASET) if CODE_DATASET else find_code()
    # /kaggle/input is read-only and pip needs a writable copy; /tmp keeps the code out of
    # the saved output.
    code_dir = Path("/tmp/asr-noisy-vi")
    shutil.copytree(source, code_dir, dirs_exist_ok=True)
    os.chdir(code_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebooks]"], check=True)
version_file = code_dir / "CODE_VERSION.json"
print("code at", code_dir)
print(version_file.read_text() if version_file.exists() else "running from a git checkout")

In [ ]:
import pandas as pd

from core import SAMPLE_RATE, extract_info
from dataprep import (
    NoiseBank,
    build_dev_plan,
    build_test_plan,
    build_train_schedule,
    describe,
    download_fsd50k_traffic,
    download_musan_speech,
    materialize_pool,
    prepare_fleurs,
    prepare_vietmed,
    prepare_vivos,
    require_data,
    save_manifests,
    select_fsd50k,
    select_musan_speech,
    set_progress_interval,
    speaker_overlap,
    subsample,
    summarize_manifest,
    write_dataset_info,
    write_licenses,
    write_noisy_set,
)

pd.set_option("display.width", 200)

## Configuration

Every parameter is written to `DATASET.json`. Changing one means building a new dataset
version: the model notebooks of the whole team must use the same one.

In [ ]:
DATA_ROOT = Path("/kaggle/working/asr-data") if ON_KAGGLE else Path("../asr-data").resolve()
RAW_DIR = Path("/tmp/asr-raw") if ON_KAGGLE else Path("../asr-raw").resolve()
MANIFEST_DIR = DATA_ROOT / "manifests"
VERSION = "v1"

CONFIG = {
    "seed": 42,
    "vivos_dev_speakers_per_gender": 2,
    "vietmed_test_utterances": 1000,   # VietMed test is subsampled, stratified by rec_condition
    "musan_subsets": ["librivox"],
    "musan_test_fraction": 0.3,
    "test_snrs": [20, 10, 5, 0],
    "dev_clean_prob": 0.25,
    "train_clean_prob": 0.25,
    "train_snr_range": [0.0, 20.0],
    "schedule_epochs": 10,             # more than any fine-tuning run will use
}
SEED = CONFIG["seed"]

LIMIT = None  # dry run: e.g. 20 utterances per split and 20 noise files per split
PROGRESS_INTERVAL_S = 60  # one progress line per long step every minute; None = silent

# Optional: already attached copies (Kaggle inputs). None = download what is needed.
FSD50K_ROOT = None  # folder with FSD50K.ground_truth/, FSD50K.dev_audio/, FSD50K.eval_audio/
MUSAN_ROOT = None   # folder with speech/librivox/

for d in (DATA_ROOT, RAW_DIR, MANIFEST_DIR):
    d.mkdir(parents=True, exist_ok=True)
for d in (DATA_ROOT, RAW_DIR):
    total, used, free = shutil.disk_usage(d)
    print(f"{d}: {free / 1e9:.0f} GB free of {total / 1e9:.0f} GB")
set_progress_interval(PROGRESS_INTERVAL_S)
print("LIMIT =", LIMIT)

## 1. Clean speech

FLEURS and VietMed keep their official splits (FLEURS `validation` → `dev`). VIVOS has no
dev split: 2 male + 2 female train speakers are held out. Transcripts are normalized
(lowercase, numbers as words, no punctuation); utterances outside 0.5–30 s are dropped.

In [ ]:
fleurs = save_manifests(prepare_fleurs(DATA_ROOT, limit=LIMIT, raw_dir=RAW_DIR),
                        MANIFEST_DIR, "fleurs")
print("FLEURS:", len(fleurs), "kept")

In [ ]:
vivos_rows, vivos_dev_speakers = prepare_vivos(
    DATA_ROOT, dev_speakers_per_gender=CONFIG["vivos_dev_speakers_per_gender"], seed=SEED,
    limit=LIMIT, raw_dir=RAW_DIR)
vivos = save_manifests(vivos_rows, MANIFEST_DIR, "vivos")
print("VIVOS:", len(vivos), "kept; dev speakers:", sorted(vivos_dev_speakers))

In [ ]:
vietmed = save_manifests(prepare_vietmed(DATA_ROOT, limit=LIMIT, raw_dir=RAW_DIR),
                         MANIFEST_DIR, "vietmed")
print("VietMed:", len(vietmed), "kept")

In [ ]:
names = ["fleurs", "vivos", "vietmed"]
clean = pd.concat([pd.read_csv(MANIFEST_DIR / f"clean_{n}.csv") for n in names],
                  ignore_index=True)
assert clean["utt_id"].is_unique, clean[clean["utt_id"].duplicated(keep=False)].head()
clean.to_csv(MANIFEST_DIR / "clean.csv", index=False)

dropped = [pd.read_csv(MANIFEST_DIR / f"dropped_{n}.csv") for n in names]
dropped = [d for d in dropped if len(d)]  # a dataset with nothing dropped has an empty file
dropped = pd.concat(dropped, ignore_index=True) if dropped else pd.DataFrame()
if len(dropped):
    print(dropped.groupby(["dataset", "split", "drop_reason"]).size().to_string(), "\n")
summarize_manifest(clean)

Leakage check: no speaker (FLEURS: no sentence) may be shared between splits. All
`shared` values should be 0.

In [ ]:
pd.concat([
    speaker_overlap(clean[clean["dataset"] != "fleurs"], key="speaker"),
    speaker_overlap(clean[clean["dataset"] == "fleurs"], key="sentence_id"),
], ignore_index=True)

In [ ]:
sample = clean.groupby("dataset").sample(n=3, random_state=SEED)
qc = pd.DataFrame([vars(extract_info(DATA_ROOT / p)) for p in sample["path"]])
assert (qc["orig_sr"] == SAMPLE_RATE).all() and (qc["orig_channels"] == 1).all()
assert ((qc["duration_s"] - sample["duration_s"].to_numpy()).abs() < 1e-3).all()
print("clean audio QC passed")

## 2. Noise pools

- **Traffic**: FSD50K road-traffic clips without speech or music labels. FSD50K `dev`
  clips are for training, `eval` clips for testing. Only these clips are downloaded.
  Clips with more than 10% digital silence are dropped (the cell prints how many).
- **Voices** (for babble): MUSAN `speech/librivox`, 30% of the files for testing.

The files used are copied into `noise/` as 16 kHz WAV, so the dataset is self-contained.

In [ ]:
if FSD50K_ROOT is not None:
    traffic_src = select_fsd50k(FSD50K_ROOT)
else:
    traffic_src = download_fsd50k_traffic(RAW_DIR, limit=LIMIT)
traffic = materialize_pool(traffic_src, DATA_ROOT, "traffic")
traffic.to_csv(MANIFEST_DIR / "noise_traffic.csv", index=False)
print("traffic clips:", traffic["split"].value_counts().to_dict())

In [ ]:
# Without MUSAN_ROOT this streams the 11 GB OpenSLR archive and keeps only the voices.
musan_root = MUSAN_ROOT or download_musan_speech(RAW_DIR, subsets=CONFIG["musan_subsets"],
                                                 limit=LIMIT)
voices_src = select_musan_speech(musan_root, subsets=CONFIG["musan_subsets"],
                                 test_fraction=CONFIG["musan_test_fraction"], seed=SEED)
voices = materialize_pool(voices_src, DATA_ROOT, "voices")
voices.to_csv(MANIFEST_DIR / "noise_voices.csv", index=False)
print("voices:", voices.groupby(["split", "gender"]).size().to_dict())

In [ ]:
pools = {"traffic": traffic, "babble": voices}
pd.concat([traffic.assign(kind="traffic"), voices.assign(kind="voices")]).groupby(
    ["kind", "split"]).agg(files=("noise_id", "size"),
                           hours=("duration_s", lambda d: round(d.sum() / 3600, 2)),
                           mean_s=("duration_s", lambda d: round(d.mean(), 1)))

## 3. Noisy test and dev sets, training schedule

- **Test**: each utterance × (clean + traffic/babble × 20/10/5/0 dB), the same noise segment
  at every SNR, noise from the **test** pools. VietMed test is subsampled to 1000
  utterances. Files are written to `noisy/` and their SNR is measured.
- **Dev**: one condition per utterance (25% clean), noise from the **train** pools.
- **Training schedule**: one condition per utterance per epoch (25% clean, SNR uniform in
  0–20 dB), noise from the **train** pools. No audio is written: training mixes online.

In [ ]:
bank = NoiseBank(traffic, voices, data_root=DATA_ROOT)

test = clean[clean["split"] == "test"]
vietmed_test = test[test["dataset"] == "vietmed"]
test = pd.concat([test[test["dataset"] != "vietmed"],
                  subsample(vietmed_test, CONFIG["vietmed_test_utterances"],
                            by="rec_condition", seed=SEED)], ignore_index=True)
test_plan = build_test_plan(test, pools, snrs=CONFIG["test_snrs"], seed=SEED)

parts = []
for name, part in test_plan.groupby("dataset", sort=False):
    parts.append(write_noisy_set(part, bank, DATA_ROOT, label=f"noisy test {name}"))
test_plan = pd.concat(parts, ignore_index=True)
test_plan.to_csv(MANIFEST_DIR / "test_plan.csv", index=False)

In [ ]:
dev = clean[clean["split"] == "dev"]
dev_plan = build_dev_plan(dev, pools, snrs=CONFIG["test_snrs"],
                          clean_prob=CONFIG["dev_clean_prob"], seed=SEED)
dev_plan = write_noisy_set(dev_plan, bank, DATA_ROOT, label="noisy dev")
dev_plan.to_csv(MANIFEST_DIR / "dev_plan.csv", index=False)

In [ ]:
# QC: every written file within 0.5 dB of its target SNR, and none clipped.
noisy = pd.concat([test_plan, dev_plan])
noisy = noisy[noisy["noise_type"] != "clean"]
error = (noisy["snr_measured"].astype(float) - noisy["snr_db"].astype(float)).abs()
print(f"{len(noisy)} noisy files, SNR error max {error.max():.3f} dB, "
      f"mean {error.mean():.3f} dB, clipped {noisy['has_clipping'].astype(bool).sum()}")
assert (error < 0.5).all(), noisy[error >= 0.5].head()
assert not noisy["has_clipping"].astype(bool).any()
test_plan.groupby(["dataset", "condition"]).size().unstack()

In [ ]:
train = clean[clean["split"] == "train"]
schedule = build_train_schedule(train, pools, epochs=CONFIG["schedule_epochs"],
                                clean_prob=CONFIG["train_clean_prob"],
                                snr_range=tuple(CONFIG["train_snr_range"]), seed=SEED)
schedule.to_csv(MANIFEST_DIR / "train_schedule.csv", index=False)
print(f"schedule: {len(train)} utterances × {CONFIG['schedule_epochs']} epochs")
schedule[schedule["epoch"] == 1]["noise_type"].value_counts(normalize=True).round(3)

## 4. Finalize

`DATASET.json` is written last: its presence marks a complete dataset, and the model
notebooks check it with `require_data`.

In [ ]:
for path in [MANIFEST_DIR / f"clean_{n}.csv" for n in names] + \
            [MANIFEST_DIR / f"dropped_{n}.csv" for n in names]:
    path.unlink(missing_ok=True)  # per-dataset copies; clean.csv holds everything

write_licenses(DATA_ROOT)
info = write_dataset_info(DATA_ROOT, {**CONFIG, "limit": LIMIT}, version=VERSION)
require_data(DATA_ROOT, check_audio=True)
print(describe(info))

In [ ]:
def folder_stats(folder):
    files = [p for p in folder.rglob("*") if p.is_file()]
    return len(files), sum(p.stat().st_size for p in files) / 1e9

rows = []
for folder in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir()):
    n, gb = folder_stats(folder)
    rows.append({"folder": folder.name, "files": n, "GB": round(gb, 2)})
stats = pd.DataFrame(rows)
print(stats.to_string(index=False))
print(f"total: {stats['files'].sum()} files, {stats['GB'].sum():.2f} GB")
if ON_KAGGLE:
    extra = [p.name for p in Path("/kaggle/working").iterdir() if p != DATA_ROOT]
    print("other items in /kaggle/working (should be empty):", extra)